# Laboratorio AWS — Predicción de temperatura de un transformador
## Notebook 05 — Model Registry y versionamiento del modelo

### Objetivo

Introducir Amazon SageMaker Model Registry para separar:

```text
modelo entrenado
       ↓
artefacto
       ↓
versión registrada
       ↓
aprobación
       ↓
posible despliegue
```

Este notebook no requiere Training Jobs ni endpoints.

El artefacto utilizado proviene del Notebook 02.

## Paso 1. Concepto de Model Registry

Model Registry permite organizar versiones de modelos dentro de un Model Group.

Para este laboratorio utilizaremos:

```text
transformadores-temperatura
        │
        └── versión del modelo
```

La ventaja es separar el artefacto físico del modelo de su ciclo de vida.

In [16]:
import json
import time
from pathlib import Path

import boto3
import joblib
import pandas as pd

BUCKET = "siemens-ml-lab-aiarch-2026"
REGION = boto3.Session().region_name or "us-east-2"

MODEL_KEY = "model/mejor_modelo.joblib"
REGISTRY_GROUP = "transformadores-temperatura"

s3 = boto3.client("s3", region_name=REGION)
sm = boto3.client("sagemaker", region_name=REGION)

print(f"Región: {REGION}")
print(f"Grupo: {REGISTRY_GROUP}")

Región: us-east-2
Grupo: transformadores-temperatura


## Paso 2. Verificar el artefacto seleccionado

El registro debe apuntar a un artefacto real almacenado en S3.

In [17]:
head = s3.head_object(Bucket=BUCKET, Key=MODEL_KEY)

MODEL_S3_URI = f"s3://{BUCKET}/{MODEL_KEY}"

print(f"Artefacto: {MODEL_S3_URI}")
print(f"Tamaño: {head['ContentLength']:,} bytes")
print(f"Última modificación: {head['LastModified']}")

Artefacto: s3://siemens-ml-lab-aiarch-2026/model/mejor_modelo.joblib
Tamaño: 1,008 bytes
Última modificación: 2026-08-12 00:33:31+00:00


## Paso 3. Crear un artefacto empaquetado para SageMaker

SageMaker Model Registry utiliza un paquete de modelo que puede referenciar un artefacto `model.tar.gz`.

Por eso crearemos una copia empaquetada del modelo de Notebook 02.

Este paso solamente trabaja con archivos y S3; no crea cómputo.

In [18]:
import tarfile

LOCAL_MODEL = Path("mejor_modelo.joblib")
PACKAGE_DIR = Path("registry_package")
PACKAGE_DIR.mkdir(exist_ok=True)

s3.download_file(BUCKET, MODEL_KEY, str(LOCAL_MODEL))

with tarfile.open("model.tar.gz", "w:gz") as tar:
    tar.add(LOCAL_MODEL, arcname="model.joblib")

REGISTRY_KEY = "model/registry/model.tar.gz"

s3.upload_file(
    "model.tar.gz",
    BUCKET,
    REGISTRY_KEY
)

REGISTRY_MODEL_URI = f"s3://{BUCKET}/{REGISTRY_KEY}"

print(f"Artefacto preparado: {REGISTRY_MODEL_URI}")

Artefacto preparado: s3://siemens-ml-lab-aiarch-2026/model/registry/model.tar.gz


## Paso 4. Crear o verificar el Model Group

Un Model Group representa una familia de modelos para el mismo problema
de negocio.

En este caso utilizaremos:

`transformadores-temperatura`

El código está diseñado para poder ejecutarse varias veces.

Si el Model Group ya existe, se reutiliza.

Si no existe, se crea.

Esta estrategia evita errores cuando el notebook se vuelve a ejecutar.

In [19]:
# ==========================================
# Paso 4. Crear o verificar Model Group
# ==========================================

try:

    # Primero verificamos si ya existe
    response = sm.describe_model_package_group(
        ModelPackageGroupName=REGISTRY_GROUP
    )

    MODEL_GROUP_ARN = response["ModelPackageGroupArn"]

    print("El Model Group ya existe.")
    print()
    print(f"Nombre: {REGISTRY_GROUP}")
    print(f"ARN:    {MODEL_GROUP_ARN}")


except sm.exceptions.ResourceNotFound:

    # Si no existe, lo creamos
    response = sm.create_model_package_group(
        ModelPackageGroupName=REGISTRY_GROUP,
        ModelPackageGroupDescription=(
            "Modelos para predicción de temperatura "
            "de transformadores eléctricos."
        )
    )

    MODEL_GROUP_ARN = response["ModelPackageGroupArn"]

    print("Model Group creado correctamente.")
    print()
    print(f"Nombre: {REGISTRY_GROUP}")
    print(f"ARN:    {MODEL_GROUP_ARN}")

El Model Group ya existe.

Nombre: transformadores-temperatura
ARN:    arn:aws:sagemaker:us-east-2:073078215254:model-package-group/transformadores-temperatura


## Paso 5. Registrar una versión del modelo

Se registra el artefacto como una versión del modelo.

El estado inicial será:

```text
PendingManualApproval
```

Esto permite explicar un flujo realista de promoción:

```text
Entrenamiento
    ↓
Registro
    ↓
Validación
    ↓
Aprobación
    ↓
Despliegue
```

In [20]:
# Imagen de inferencia de Scikit-learn.
# Se utiliza una imagen administrada como referencia para el paquete registrado.
from sagemaker.core import image_uris

inference_image = image_uris.retrieve(
    framework="sklearn",
    region=REGION,
    version="1.2-1",
    py_version="py3",
    image_scope="inference"
)

response = sm.create_model_package(
    ModelPackageGroupName=REGISTRY_GROUP,

    ModelPackageDescription=(
        "Modelo de regresión para predecir "
        "la temperatura del transformador."
    ),

    InferenceSpecification={
        "Containers": [
            {
                "Image": inference_image,
                "ModelDataUrl": REGISTRY_MODEL_URI
            }
        ],
        "SupportedContentTypes": [
            "application/json"
        ],
        "SupportedResponseMIMETypes": [
            "application/json"
        ]
    },

    ModelApprovalStatus="PendingManualApproval"
)

MODEL_PACKAGE_ARN = response["ModelPackageArn"]

print("Modelo registrado correctamente.")
print()
print(f"ARN: {MODEL_PACKAGE_ARN}")

[08/12/26 00:43:44] INFO     Defaulting to only supported image scope: cpu.                       ]8;id=2522257;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py\image_uris.py]8;;\:]8;id=2522258;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/image_uris.py#539\539]8;;\

Modelo registrado correctamente.

ARN: arn:aws:sagemaker:us-east-2:073078215254:model-package/transformadores-temperatura/3


## Paso 6. Consultar y aprobar la versión

La aprobación representa una decisión de gobierno del modelo.

En un escenario empresarial:

- `PendingManualApproval`: pendiente de validación.
- `Approved`: autorizado para despliegue.
- `Rejected`: no autorizado.

En este laboratorio aprobamos la versión para completar el ciclo
didáctico de Model Registry.

La operación de actualización utiliza el ARN de la versión registrada.

In [21]:
# ==========================================
# Paso 6. Consultar y aprobar la versión
# ==========================================

details = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN
)

print("Estado actual:")
print(details["ModelApprovalStatus"])

print()

# Aprobar la versión
sm.update_model_package(
    ModelPackageArn=MODEL_PACKAGE_ARN,
    ModelApprovalStatus="Approved",
    ApprovalDescription=(
        "Validado para el laboratorio de "
        "predicción de temperatura."
    )
)

# Consultar nuevamente
updated = sm.describe_model_package(
    ModelPackageName=MODEL_PACKAGE_ARN
)

print("Nuevo estado:")
print(updated["ModelApprovalStatus"])

Estado actual:
PendingManualApproval

Nuevo estado:
Approved


## Paso 7. Verificación final y cierre

Consultamos las versiones registradas.

El resultado conceptual es:

```text
S3 / model/mejor_modelo.joblib
              ↓
     model.tar.gz
              ↓
      Model Registry
              ↓
       Model Group
              ↓
        Model Version
              ↓
          Approved
```

Esto prepara el laboratorio para MLOps.

> No fue necesario ejecutar un Training Job ni mantener un endpoint para aprender el ciclo de gobierno del modelo.

In [22]:
# ==========================================
# Paso 7. Verificación final
# ==========================================

response = sm.list_model_packages(
    ModelPackageGroupName=REGISTRY_GROUP,
    SortBy="CreationTime",
    SortOrder="Descending"
)

model_packages = response.get(
    "ModelPackageSummaryList",
    []
)

print("Modelos registrados:")
print()

for item in model_packages:

    print(
        f"Versión: {item.get('ModelPackageVersion')} | "
        f"Estado: {item.get('ModelPackageStatus')} | "
        f"Aprobación: {item.get('ModelApprovalStatus')}"
    )

Modelos registrados:

Versión: 3 | Estado: Completed | Aprobación: Approved
Versión: 2 | Estado: Completed | Aprobación: Approved
Versión: 1 | Estado: Completed | Aprobación: Approved
